# Machine Learning – Class Assignment
## Predicting Loan Default using a Decision Tree Classifier
**Name:** _your name_  |  **Roll No:** _your roll no_

### Setup (Google Colab only)
Run this cell and upload `loan_default.csv` when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()   # choose loan_default.csv

## Step 1: Import libraries and load the data

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns
import warnings; warnings.filterwarnings('ignore')
df = pd.read_csv('loan_default.csv')
print(df.shape); df.info(); print(df.head(10))

**Write down – Answer:** The dataset has **1012 rows and 18 columns**. Numeric columns (11): age, income, loan_amount, credit_score, employment_years, debt_to_income, existing_loans, dependents, bank_balance, interest_rate, previous_defaults. Categorical (text) columns: employment_type, marital_status, education, loan_purpose, city_tier, plus the target loan_default. customer_id is just an identifier.

## Step 2: Understand the data

In [ ]:
print(df.describe())
for c in df.select_dtypes('object').columns.drop('customer_id'):
    print(df[c].value_counts(dropna=False), '\n')
sns.countplot(x='loan_default', data=df); plt.title('Loan default'); plt.show()
print(df['loan_default'].value_counts(normalize=True))

**Write down – Answer:** The target is **imbalanced**: about 81.5% No (825) vs 18.5% Yes (187). This matters because a model can reach ~81% accuracy by always predicting “No” while catching no defaulters, so accuracy alone is misleading and recall / F1 / ROC-AUC must be checked.

## Step 3: Clean the data

In [ ]:
print(df.isnull().sum())
df['education'] = df['education'].fillna(df['education'].mode()[0])
print('Duplicate rows:', df.duplicated().sum())
df = df.drop_duplicates()
print('Negative ages:', (df['age']<0).sum(), '| Negative incomes:', (df['income']<0).sum())
df.loc[df['age']<0,'age'] = df['age'].abs()
df.loc[df['income']<0,'income'] = df['income'].abs()
df = df.drop_duplicates()
print('Duplicate customer_ids left:', df['customer_id'].duplicated().sum())
print('Shape after cleaning:', df.shape)
print(df[['age','income']].min())

**Write down – Answer:** Only **education** had missing values (61). I filled them with the **most frequent value (mode)** because it is a categorical column, so a mean cannot be used, and this keeps all 61 rows instead of losing data. I also found **11 duplicate rows** (dropped) and **5 negative ages and 5 negative incomes**, which are impossible; they looked like sign typos (e.g., age −22 was a duplicate of age 22), so I converted them with abs(). After that one more duplicate appeared and was dropped, leaving **1000 rows**.

## Step 4: Explore the data visually

In [ ]:
fig, ax = plt.subplots(1,2, figsize=(12,4))
sns.boxplot(x='loan_default', y='credit_score', data=df, ax=ax[0])
sns.histplot(data=df, x='credit_score', hue='loan_default', kde=True, ax=ax[1]); plt.show()
fig, ax = plt.subplots(1,2, figsize=(12,4))
for a,col in zip(ax,['city_tier','employment_type']):
    (df.groupby(col)['loan_default'].apply(lambda s:(s=='Yes').mean()).plot(kind='bar', ax=a, title=f'Default rate by {col}')); a.set_ylabel('default rate')
plt.show()
num = df.select_dtypes('number')
plt.figure(figsize=(10,8)); sns.heatmap(num.assign(loan_default=df['loan_default'].map({'Yes':1,'No':0})).corr(), annot=True, fmt='.2f', cmap='coolwarm'); plt.show()
print(num.assign(loan_default=df['loan_default'].map({'Yes':1,'No':0})).corr()['loan_default'].sort_values())
print(df.groupby('city_tier')['loan_default'].apply(lambda s:(s=='Yes').mean()))
print(df.groupby('employment_type')['loan_default'].apply(lambda s:(s=='Yes').mean()))

**Write down – Answer:** **credit_score** (defaulters have clearly lower scores; correlation about −0.30) and **interest_rate** (defaulters pay higher rates; correlation about +0.24) separate the classes best. loan_amount, debt_to_income and previous_defaults help a little. Default rate barely differs by city_tier (17–19%); Contract workers have a somewhat higher default rate (~24%) than Salaried (~17%).

## Step 5: Prepare the data for modelling

In [ ]:
df = df.drop(columns=['customer_id'])
df['loan_default'] = df['loan_default'].map({'Yes':1,'No':0})
df = pd.get_dummies(df, drop_first=True)
X = df.drop('loan_default', axis=1); y = df['loan_default']
print(X.shape); print(X.columns.tolist())

**Write down – Answer:** scikit-learn trees do mathematical comparisons (feature ≤ threshold) and work only on numbers, so text such as “Salaried” or “Tier 2” must first be encoded into numeric columns (one-hot encoding with get_dummies).

## Step 6: Split into training and test sets

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print('Train:', X_train.shape, '| Test:', X_test.shape)

**Write down – Answer:** The test set simulates new, unseen customers. If we trained on it, the model would just memorise those rows and our score would be over-optimistic; keeping it aside gives an honest estimate of real-world performance.

## Step 7: Train a Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
dt = DecisionTreeClassifier(random_state=42).fit(X_train, y_train)
print('Train accuracy:', dt.score(X_train, y_train)); print('Test accuracy :', dt.score(X_test, y_test))

**Write down – Answer:** Training accuracy is **1.00** but test accuracy is only **~0.79**. This big gap shows **overfitting**: the fully grown tree memorised the training data, noise included, and does not generalise well.

## Step 8: Evaluate the model properly

In [ ]:
from sklearn.metrics import (confusion_matrix, ConfusionMatrixDisplay, classification_report, roc_auc_score, accuracy_score, recall_score, f1_score)
y_pred = dt.predict(X_test)
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred), display_labels=['No','Yes']).plot(cmap='Blues'); plt.show()
print(classification_report(y_test, y_pred))
print('ROC-AUC:', roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]))

**Write down – Answer:** Approving a loan that defaults is worse: the bank loses the principal, whereas rejecting a good customer only loses some interest income. This makes **recall for the “Yes” (default) class** the most important metric (along with ROC-AUC). The untuned tree has recall of only ~0.27 for defaulters and ROC-AUC ~0.59, so it misses most defaulters despite ~79% accuracy.

## Step 9: Control overfitting by tuning

In [ ]:
rows = []
for d in [3,5,7,10]:
    m = DecisionTreeClassifier(max_depth=d, random_state=42).fit(X_train, y_train)
    rows.append((d, m.score(X_train,y_train), m.score(X_test,y_test)))
print(pd.DataFrame(rows, columns=['max_depth','train_acc','test_acc']).round(3))
from sklearn.model_selection import GridSearchCV
param_grid = {'max_depth':[3,5,7,10,None], 'min_samples_leaf':[1,5,10,20], 'criterion':['gini','entropy']}
grid = GridSearchCV(DecisionTreeClassifier(random_state=42), param_grid, cv=5, scoring='roc_auc', n_jobs=-1).fit(X_train, y_train)
print('Best params:', grid.best_params_); print('Best CV ROC-AUC:', round(grid.best_score_,3))
best = grid.best_estimator_
print('Train acc:', best.score(X_train,y_train), '| Test acc:', best.score(X_test,y_test))
print(classification_report(y_test, best.predict(X_test)))
print('Test ROC-AUC:', roc_auc_score(y_test, best.predict_proba(X_test)[:,1]))

**Write down – Answer:** Among max_depth 3, 5, 7, 10, **max_depth = 5** gave the best test accuracy (~0.82). Deeper trees (7, 10) raise training accuracy but lower test accuracy, which is overfitting. GridSearchCV (scored on ROC-AUC) chose criterion = entropy, max_depth = 10, min_samples_leaf = 20: the training accuracy dropped from 1.00 to ~0.85 (much less overfitting), but test accuracy (~0.78) and ROC-AUC (~0.59) did **not** improve over Step 7, and recall fell. With only 1000 rows and 200 test rows, results are noisy, and the tree itself is a weak model for this data.

## Step 10: Compare with a baseline model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler().fit(X_train)
lr = LogisticRegression(max_iter=1000, random_state=42).fit(scaler.transform(X_train), y_train)
def metrics(name, model, Xt):
    p = model.predict(Xt); pr = model.predict_proba(Xt)[:,1]
    return {'Model':name,'Accuracy':accuracy_score(y_test,p),'Recall':recall_score(y_test,p),'F1':f1_score(y_test,p),'ROC-AUC':roc_auc_score(y_test,pr)}
comp = pd.DataFrame([metrics('Logistic Regression', lr, scaler.transform(X_test)), metrics('Decision Tree (untuned)', dt, X_test), metrics('Decision Tree (tuned)', best, X_test)]).set_index('Model').round(3)
print(comp)

**Write down – Answer:** I would recommend **Logistic Regression**. It has the best accuracy (~0.84), the best F1 (~0.36) and by far the best ROC-AUC (~0.78 vs ~0.59 for both trees), so it ranks risky customers much better, and it is simple and easy to explain. Its recall (~0.24) is still low, so in practice the bank should lower the decision threshold to catch more defaulters.

## Step 11: Interpret the model

In [ ]:
pd.Series(best.feature_importances_, index=X.columns).nlargest(10).sort_values().plot(kind='barh', title='Top 10 feature importances'); plt.show()
print(pd.Series(best.feature_importances_, index=X.columns).nlargest(5))
plt.figure(figsize=(22,10)); plot_tree = __import__('sklearn.tree',fromlist=['plot_tree']).plot_tree
plot_tree(best, max_depth=3, feature_names=X.columns, class_names=['No','Yes'], filled=True, fontsize=9); plt.show()
from sklearn.tree import export_text
print(export_text(best, feature_names=list(X.columns), max_depth=3))

**Write down – Answer:** Top five features: **credit_score, employment_years, interest_rate, loan_amount, debt_to_income**. credit_score and interest_rate match Step 4; loan_amount and debt_to_income were weaker but positive in the correlation map. employment_years did not stand out in Step 4, so the tree may be using it for noise. Two rules in plain English: (1) *If credit_score > 748, the tree predicts the customer will not default* (very high scores are safe). (2) *If credit_score ≤ 673 and loan_amount > 234,000 and the interest rate > 10.45% and debt-to-income ≤ 0.33, the customer is predicted not to default; with debt-to-income above 0.33 the tree goes deeper to decide* — i.e., low credit score plus a large, expensive loan and heavy debt is the risky zone. *(Check the printed tree rules in your own run and quote two that you can see.)*

## Step 12: Conclusion

**Summary for the bank manager:** We built and tested models that predict loan default using 1000 cleaned customer records. A decision tree fitted without limits memorised the training data and performed poorly on new customers (ROC-AUC ~0.59), and tuning reduced overfitting but did not fix it. Logistic Regression performed best (accuracy ~0.84, ROC-AUC ~0.78) and is easy to explain. The most useful signals are a low credit score and a high interest rate. **Does well:** correctly identifies most good customers (~90% recall for non-defaulters). **Limitations:** it catches only about a quarter of true defaulters, because only 18% of the data are defaults and the dataset is small. **Improvement:** collect more data and use class weighting / a lower decision threshold (or a Random Forest) to raise recall on defaulters.

## Extra: Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf = RandomForestClassifier(n_estimators=300, random_state=42).fit(X_train, y_train)
print(pd.concat([comp, pd.DataFrame([metrics('Random Forest', rf, X_test)]).set_index('Model').round(3)]))

A Random Forest (ROC-AUC ~0.73, accuracy ~0.83) beats the tuned tree (ROC-AUC ~0.59) because averaging many trees reduces overfitting, but its recall for defaulters (~0.11) is still low and Logistic Regression remains best here.